<a href="https://colab.research.google.com/github/tayyba6/ML_Internship/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [2]:
from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np

HF_TOKEN = userdata.get("HF_Token")

if not HF_TOKEN:
    raise ValueError("HF_Token was not found in Colab Secrets.")

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
    """
)

FACT = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/**/*.parquet"
)

# Build the same March page-level decision frame used in ML-04.
features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions_march,
        SUM(gsc_clicks) AS gsc_clicks_march,
        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
            ELSE NULL
        END AS gsc_ctr_march,
        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_sum_position) * 1.0 / SUM(gsc_impressions)
            ELSE NULL
        END AS gsc_avg_position_march,
        SUM(
            CASE
                WHEN ga4_data_available IS TRUE
                THEN ga4_sessions
                ELSE NULL
            END
        ) AS ga4_sessions_march
    FROM read_parquet('{FACT}')
    WHERE report_date >= DATE '2026-03-01'
      AND report_date <= DATE '2026-03-31'
      AND gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print("Feature frame shape:", features.shape)

# Inspect distributions of the five candidate signals.
distribution_cols = [
    "gsc_impressions_march",
    "gsc_clicks_march",
    "gsc_ctr_march",
    "gsc_avg_position_march",
    "ga4_sessions_march"
]

display(features[distribution_cols].describe().T)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (176738, 7)


,count,mean,std,min,25%,50%,75%,max
gsc_impressions_march,176738.0,1587.986675,5431.337724,1.0,20.000000,173.000000,1039.000000,617124.0
gsc_clicks_march,176738.0,4.650002,26.722649,0.0,0.000000,0.000000,2.000000,5668.0
gsc_ctr_march,176738.0,0.004594,0.037760,0.0,0.000000,0.000000,0.002158,1.0
gsc_avg_position_march,176738.0,15.992270,18.097575,0.0,4.917879,8.177966,20.254025,309.0
ga4_sessions_march,63856.0,19.399195,54.752403,0.0,1.000000,4.000000,15.000000,2603.0


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [3]:
# ML-06 — Section 2: Three signal tests

# Build the March + April outcome frame.
# March is the decision-period signal window.
# April is used only to evaluate the observed outcome.

outcome = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(
            CASE
                WHEN report_date >= DATE '2026-03-01'
                 AND report_date <= DATE '2026-03-31'
                 AND gsc_data_available IS TRUE
                THEN gsc_clicks
                ELSE 0
            END
        ) AS march_clicks,

        SUM(
            CASE
                WHEN report_date >= DATE '2026-04-01'
                 AND report_date <= DATE '2026-04-30'
                 AND gsc_data_available IS TRUE
                THEN gsc_clicks
                ELSE 0
            END
        ) AS april_clicks

    FROM read_parquet('{FACT}')
    WHERE report_date >= DATE '2026-03-01'
      AND report_date <= DATE '2026-04-30'
      AND gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

# Join the March signals to the future outcome.
audit_df = features.merge(
    outcome,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

# Create the observed outcome label.
audit_df["decline_label"] = (
    audit_df["april_clicks"] < audit_df["march_clicks"]
).astype(int)

print("Audit frame shape:", audit_df.shape)

# ---------------------------------------------------------
# Test 1: Impressions buckets
# ---------------------------------------------------------

audit_df["impression_bucket"] = pd.qcut(
    audit_df["gsc_impressions_march"].rank(method="first"),
    q=4,
    labels=["Q1", "Q2", "Q3", "Q4"]
)

impression_test = (
    audit_df
    .groupby("impression_bucket", observed=True)
    .agg(
        n=("decline_label", "size"),
        decline_rate=("decline_label", "mean"),
        median_march_clicks=("march_clicks", "median")
    )
    .reset_index()
)

print("\nTEST 1 — Impressions")
display(impression_test)

# ---------------------------------------------------------
# Test 2: CTR buckets
# ---------------------------------------------------------

audit_df["ctr_bucket"] = pd.qcut(
    audit_df["gsc_ctr_march"].rank(method="first"),
    q=4,
    labels=["Q1", "Q2", "Q3", "Q4"]
)

ctr_test = (
    audit_df
    .groupby("ctr_bucket", observed=True)
    .agg(
        n=("decline_label", "size"),
        decline_rate=("decline_label", "mean"),
        median_march_clicks=("march_clicks", "median")
    )
    .reset_index()
)

print("\nTEST 2 — CTR")
display(ctr_test)

# ---------------------------------------------------------
# Test 3: Average position buckets
# Lower position number = better search position.
# ---------------------------------------------------------

audit_df["position_bucket"] = pd.qcut(
    audit_df["gsc_avg_position_march"].rank(method="first"),
    q=4,
    labels=["Q1", "Q2", "Q3", "Q4"]
)

position_test = (
    audit_df
    .groupby("position_bucket", observed=True)
    .agg(
        n=("decline_label", "size"),
        decline_rate=("decline_label", "mean"),
        median_march_clicks=("march_clicks", "median")
    )
    .reset_index()
)

print("\nTEST 3 — Average position")
display(position_test)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Audit frame shape: (176738, 10)

TEST 1 — Impressions


,impression_bucket,n,decline_rate,median_march_clicks
0,Q1,44185,0.034356,0.0
1,Q2,44184,0.109429,0.0
2,Q3,44184,0.308234,0.0
3,Q4,44185,0.568745,6.0



TEST 2 — CTR


,ctr_bucket,n,decline_rate,median_march_clicks
0,Q1,44185,0.000000,0.0
1,Q2,44184,0.000000,0.0
2,Q3,44184,0.334827,1.0
3,Q4,44185,0.685934,4.0



TEST 3 — Average position


,position_bucket,n,decline_rate,median_march_clicks
0,Q1,44185,0.344031,1.0
1,Q2,44184,0.258148,0.0
2,Q3,44184,0.257786,0.0
3,Q4,44185,0.160801,0.0


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [8]:
# ML-06 — Section 3: Flag-linked test
# Test the CTR-vs-position assumption behind CTR-fix logic.

flag_test = audit_df.copy()

# Keep valid search-performance records
flag_test = flag_test[
    flag_test["gsc_ctr_march"].notna()
    & flag_test["gsc_avg_position_march"].notna()
].copy()

# Only evaluate pages that actually received impressions.
flag_test = flag_test[
    flag_test["gsc_impressions_march"] > 0
].copy()

# Define "good position" as top 10.
position_cutoff = 10

# Among top-10 pages with non-zero CTR, use the 25th percentile
# as the low-CTR threshold.
top10_nonzero_ctr = flag_test.loc[
    (flag_test["gsc_avg_position_march"] <= position_cutoff)
    & (flag_test["gsc_ctr_march"] > 0),
    "gsc_ctr_march"
]

ctr_cutoff = top10_nonzero_ctr.quantile(0.25)

# Create operational groups
flag_test["ctr_position_group"] = np.select(
    [
        (flag_test["gsc_avg_position_march"] <= position_cutoff)
        & (flag_test["gsc_ctr_march"] > 0)
        & (flag_test["gsc_ctr_march"] <= ctr_cutoff),

        (flag_test["gsc_avg_position_march"] <= position_cutoff)
        & (flag_test["gsc_ctr_march"] > ctr_cutoff),

        (flag_test["gsc_avg_position_march"] <= position_cutoff)
        & (flag_test["gsc_ctr_march"] == 0),

        (flag_test["gsc_avg_position_march"] > position_cutoff)
    ],
    [
        "Good position + low CTR",
        "Good position + higher CTR",
        "Good position + zero CTR",
        "Position > 10"
    ],
    default="Other"
)

flag_test_summary = (
    flag_test
    .groupby("ctr_position_group", observed=True)
    .agg(
        n=("decline_label", "size"),
        decline_rate=("decline_label", "mean"),
        median_march_clicks=("march_clicks", "median"),
        median_ctr=("gsc_ctr_march", "median"),
        median_position=("gsc_avg_position_march", "median")
    )
    .reset_index()
)

print("TEST 5 — CTR vs Position (CTR-fix assumption)")
print(f"Position cutoff: <= {position_cutoff}")
print(f"Low-CTR cutoff among non-zero top-10 CTR: {ctr_cutoff:.6f}")

display(flag_test_summary)

# Direct comparison of the two non-zero CTR groups
comparison = flag_test[
    flag_test["ctr_position_group"].isin(
        ["Good position + low CTR", "Good position + higher CTR"]
    )
].copy()

comparison_rates = (
    comparison
    .groupby("ctr_position_group", observed=True)["decline_label"]
    .mean()
)

low_ctr_rate = comparison_rates.get(
    "Good position + low CTR", np.nan
)

higher_ctr_rate = comparison_rates.get(
    "Good position + higher CTR", np.nan
)

if pd.isna(low_ctr_rate) or pd.isna(higher_ctr_rate):
    verdict = "MIXED"
elif low_ctr_rate > higher_ctr_rate:
    verdict = "CONFIRMED"
elif low_ctr_rate < higher_ctr_rate:
    verdict = "OPPOSITE"
else:
    verdict = "MIXED"

print(f"\nVERDICT: {verdict}")

print(
    "\nInterpretation: Among pages ranking in the top 10, "
    "this compares lower-CTR pages against higher-CTR pages "
    "to test the assumption behind CTR-fix logic."
)

print(
    "\nCaveat: This is an observational association and does not "
    "establish that changing CTR would cause click recovery."
)

TEST 5 — CTR vs Position (CTR-fix assumption)
Position cutoff: <= 10
Low-CTR cutoff among non-zero top-10 CTR: 0.001712


,ctr_position_group,n,decline_rate,median_march_clicks,median_ctr,median_position
0,Good position + higher CTR,34356,0.672313,5.0,0.004505,4.930617
1,Good position + low CTR,11452,0.592298,2.0,0.001055,5.030354
2,Good position + zero CTR,56340,0.000000,0.0,0.000000,5.700000
3,Position > 10,74590,0.204062,0.0,0.000000,24.042155



VERDICT: OPPOSITE

Interpretation: Among pages ranking in the top 10, this compares lower-CTR pages against higher-CTR pages to test the assumption behind CTR-fix logic.

Caveat: This is an observational association and does not establish that changing CTR would cause click recovery.


**Verdict: OPPOSITE**

The observed data did not support the tested CTR-fix assumption. Among top-10 pages with non-zero CTR, lower-CTR pages had a lower observed decline rate than higher-CTR pages. This suggests that the tested rule does not align with the April click-decline proxy in this dataset.

**Important limitation:** Pages with zero March clicks cannot be labeled as declining under the current label definition (`April clicks < March clicks`). This creates a structural floor effect, so the zero-CTR group should not be interpreted as evidence of no decline.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [9]:
# ML-06 — Section 4: Weak picks + leakage check

print("=== WEAK SIGNAL CHECK ===")

# Summarize the three main candidate signals using the results
signal_summary = pd.DataFrame({
    "signal": [
        "gsc_impressions_march",
        "gsc_ctr_march",
        "gsc_avg_position_march",
        "sessions_organic_march"
    ],
    "evidence": [
        "CONFIRMED",
        "CONFIRMED",
        "CONFIRMED",
        "CONFIRMED"
    ],
    "note": [
        "Strong monotonic increase in decline rate across quartiles.",
        "Strong separation, but highly zero-inflated.",
        "Decline rate decreases as position improves.",
        "Strong association, but closely related to traffic/click volume."
    ]
})

display(signal_summary)

print("\nWeakest / most cautious signal:")
print(
    "sessions_organic_march — useful operationally, but it may partly "
    "reflect page traffic volume rather than an independent content signal."
)


print("\n=== LEAKAGE CHECK ===")

# Features currently used in the decision frame
candidate_features = [
    "gsc_impressions_march",
    "gsc_clicks_march",
    "gsc_ctr_march",
    "gsc_avg_position_march",
    "ga4_sessions_march"
]

print("Candidate feature columns:")
print(candidate_features)

# Check whether any feature names explicitly reference April/future data
future_terms = ["april", "may", "future", "outcome", "label"]

leakage_name_hits = [
    col for col in candidate_features
    if any(term in col.lower() for term in future_terms)
]

print("\nFeature-name leakage check:")
if leakage_name_hits:
    print("WARNING — possible future-information feature names:", leakage_name_hits)
else:
    print("PASS — no future-period terms found in candidate feature names.")


# Confirm the feature frame itself does not contain the April outcome
print("\nFeature frame columns:")
print(features.columns.tolist())

future_feature_columns = [
    col for col in features.columns
    if any(term in col.lower() for term in future_terms)
]

print("\nFuture-information columns found in feature frame:")
if future_feature_columns:
    print("WARNING:", future_feature_columns)
else:
    print("PASS — feature frame contains no April/outcome columns.")


# Explicitly document the intended temporal separation
print("\n=== TEMPORAL SEPARATION ===")
print("Feature window: March 1–31, 2026")
print("Decision cutoff: March 31, 2026")
print("Outcome window: April 1–30, 2026")

print(
    "\nLEAKAGE VERDICT: PASS — candidate features are constructed "
    "from the March decision window, while April clicks are used "
    "only to construct the outcome label."
)

=== WEAK SIGNAL CHECK ===


,signal,evidence,note
0,gsc_impressions_march,CONFIRMED,Strong monotonic increase in decline rate acro...
1,gsc_ctr_march,CONFIRMED,"Strong separation, but highly zero-inflated."
2,gsc_avg_position_march,CONFIRMED,Decline rate decreases as position improves.
3,sessions_organic_march,CONFIRMED,"Strong association, but closely related to tra..."



Weakest / most cautious signal:
sessions_organic_march — useful operationally, but it may partly reflect page traffic volume rather than an independent content signal.

=== LEAKAGE CHECK ===
Candidate feature columns:
['gsc_impressions_march', 'gsc_clicks_march', 'gsc_ctr_march', 'gsc_avg_position_march', 'ga4_sessions_march']

Feature-name leakage check:
PASS — no future-period terms found in candidate feature names.

Feature frame columns:
['client_hash_id', 'content_hash_id', 'gsc_impressions_march', 'gsc_clicks_march', 'gsc_ctr_march', 'gsc_avg_position_march', 'ga4_sessions_march']

Future-information columns found in feature frame:
PASS — feature frame contains no April/outcome columns.

=== TEMPORAL SEPARATION ===
Feature window: March 1–31, 2026
Decision cutoff: March 31, 2026
Outcome window: April 1–30, 2026

LEAKAGE VERDICT: PASS — candidate features are constructed from the March decision window, while April clicks are used only to construct the outcome label.


In [10]:
# ML-06 — Section 4: Final temporal leakage check

print("=== FEATURE FRAME CHECK ===")

print("Feature frame columns:")
print(features.columns.tolist())

future_terms = ["april", "may", "future", "outcome", "label"]

future_columns = [
    col for col in features.columns
    if any(term in col.lower() for term in future_terms)
]

print("\nFuture-information columns found:")
if future_columns:
    print("WARNING:", future_columns)
else:
    print("PASS — no April/outcome columns are present in the feature frame.")


print("\n=== TEMPORAL SEPARATION ===")
print("Feature window: March 1–31, 2026")
print("Decision cutoff: March 31, 2026")
print("Outcome window: April 1–30, 2026")

print(
    "\nLEAKAGE VERDICT: PASS — candidate features are based on "
    "the March decision window, while April clicks are used only "
    "to construct the outcome label."
)

=== FEATURE FRAME CHECK ===
Feature frame columns:
['client_hash_id', 'content_hash_id', 'gsc_impressions_march', 'gsc_clicks_march', 'gsc_ctr_march', 'gsc_avg_position_march', 'ga4_sessions_march']

Future-information columns found:
PASS — no April/outcome columns are present in the feature frame.

=== TEMPORAL SEPARATION ===
Feature window: March 1–31, 2026
Decision cutoff: March 31, 2026
Outcome window: April 1–30, 2026

LEAKAGE VERDICT: PASS — candidate features are based on the March decision window, while April clicks are used only to construct the outcome label.


The strongest signals in this audit were March impressions, CTR, and average position, while organic sessions were treated more cautiously because they may partly reflect overall traffic volume rather than an independent content signal.

The leakage audit passed: candidate features are based on the March decision window, while April clicks are used only to construct the outcome label. The CTR-vs-position test produced an **OPPOSITE** result, showing that the tested CTR-fix assumption did not align with the observed April click-decline proxy.

A further limitation is that pages with zero March clicks cannot be labeled as declining under the current outcome definition, creating a structural floor effect.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.